In [ ]:
# ============================================================================
# 03_silver_to_gold  (Airflow Option-B port)
# ----------------------------------------------------------------------------
# Whole-run stage (cohort_id = "ALL"). The DAG passes run_id_root + stage; the
# original as_of_date knob is kept (empty -> today UTC) so re-runs stay
# deterministic. Fully overwrites the exec.* / insights.* marts each run.
# ============================================================================

# PARAMETERS CELL ************
run_id_root = ""               # run grouping key from set_run_id
stage       = "silver_to_gold"
as_of_date  = ""               # ISO date (YYYY-MM-DD); empty -> today (UTC)


In [ ]:
# ---------------------------------------------------------------------------
# run_state skip/restart helpers (gold.control.run_state)
# Paste-in contract from include/run_state_helpers.py. The DAG never queries the
# lakehouse; each notebook self-records RUNNING -> SUCCEEDED/FAILED and self-skips
# a (run_id_root, cohort_id, stage) triple that already SUCCEEDED.
# ---------------------------------------------------------------------------
from datetime import datetime, timezone

RUN_STATE_TABLE = "lh_synthea_gold.control.run_state"


def already_succeeded(run_id_root, cohort_id, stage):
    """True if this (run_id_root, cohort_id, stage) already completed."""
    if not run_id_root or not spark.catalog.tableExists(RUN_STATE_TABLE):
        return False
    df = spark.sql(
        f"""
        SELECT 1 FROM {RUN_STATE_TABLE}
        WHERE run_id_root = '{run_id_root}'
          AND cohort_id   = '{cohort_id}'
          AND stage       = '{stage}'
          AND status      = 'SUCCEEDED'
        LIMIT 1
        """
    )
    return df.count() > 0


def mark(run_id_root, cohort_id, stage, status, error=None):
    """Idempotent UPSERT of a run_state row for this triple via MERGE."""
    if not run_id_root or not spark.catalog.tableExists(RUN_STATE_TABLE):
        return
    now = datetime.now(timezone.utc)
    err = (error or "").replace("'", "''")[:4000]
    spark.sql(
        f"""
        MERGE INTO {RUN_STATE_TABLE} AS t
        USING (
            SELECT
                '{run_id_root}' AS run_id_root,
                '{cohort_id}'   AS cohort_id,
                '{stage}'       AS stage,
                '{status}'      AS status,
                TIMESTAMP('{now.isoformat()}') AS ts,
                '{err}'         AS error
        ) AS s
        ON  t.run_id_root = s.run_id_root
        AND t.cohort_id   = s.cohort_id
        AND t.stage       = s.stage
        WHEN MATCHED THEN UPDATE SET
            t.status   = s.status,
            t.attempt  = COALESCE(t.attempt, 0) + CASE WHEN s.status = 'RUNNING' THEN 1 ELSE 0 END,
            t.started_ts = CASE WHEN s.status = 'RUNNING' THEN s.ts ELSE t.started_ts END,
            t.ended_ts   = CASE WHEN s.status IN ('SUCCEEDED','FAILED') THEN s.ts ELSE t.ended_ts END,
            t.error      = CASE WHEN s.status = 'FAILED' THEN s.error ELSE NULL END
        WHEN NOT MATCHED THEN INSERT (
            run_id_root, cohort_id, stage, status, attempt, started_ts, ended_ts, error
        ) VALUES (
            s.run_id_root, s.cohort_id, s.stage, s.status, 1, s.ts, NULL,
            CASE WHEN s.status = 'FAILED' THEN s.error ELSE NULL END
        )
        """
    )

import json

# whole-run stage -> single run_state row keyed by cohort_id = "ALL"
cohort_id = "ALL"

if already_succeeded(run_id_root, cohort_id, stage):
    mssparkutils.notebook.exit(json.dumps({"status": "SKIPPED", "run_id_root": run_id_root}))
mark(run_id_root, cohort_id, stage, "RUNNING")


In [ ]:
# The original stage body is wrapped so that a thrown exception is recorded
# as FAILED in run_state, while a normal/early return is recorded SUCCEEDED.
def _run_body():
    # CODE CELL ******************
    import datetime as dt
    from pyspark.sql import SparkSession, functions as F, Window

    spark = SparkSession.builder.getOrCreate()

    if not as_of_date:
        as_of_date = dt.date.today().isoformat()

    print(f"[gold] as_of_date={as_of_date}")

    SILVER = "lh_synthea_silver.core"
    GOLD   = "lh_synthea_gold"

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}.exec")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}.insights")

    # ---------------------------------------------------------------------------
    # Source views — clip to as_of_date so re-runs are deterministic.
    # ---------------------------------------------------------------------------
    encounters = (spark.table(f"{SILVER}.encounters")
                       .filter(F.to_date("START") <= F.lit(as_of_date)))

    patients   = spark.table(f"{SILVER}.patients")
    claims     = spark.table(f"{SILVER}.claims")
    claims_tx  = spark.table(f"{SILVER}.claims_transactions")
    conditions = spark.table(f"{SILVER}.conditions")
    observations = spark.table(f"{SILVER}.observations")

    # ---------------------------------------------------------------------------
    # exec.encounter_volumes
    #   Daily count of encounters by class + cohort.
    # ---------------------------------------------------------------------------
    encounter_volumes = (encounters
        .withColumn("date", F.to_date("START"))
        .groupBy("date", "cohort_id", F.col("ENCOUNTERCLASS").alias("encounter_class"))
        .agg(F.count("*").alias("encounters"))
        .orderBy("date", "cohort_id", "encounter_class"))

    (encounter_volumes.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.exec.encounter_volumes"))
    print("[gold] wrote exec.encounter_volumes")

    # ---------------------------------------------------------------------------
    # exec.readmissions_30d
    #   Index admissions = inpatient encounters discharged on date D.
    #   Readmits = same patient with an inpatient admission within 30 days
    #   following D (excluding the index encounter itself).
    # ---------------------------------------------------------------------------
    inpt = (encounters
            .filter(F.col("ENCOUNTERCLASS") == "inpatient")
            .select(
                F.col("Id").alias("enc_id"),
                F.col("PATIENT").alias("patient"),
                "cohort_id",
                F.to_date("START").alias("admit_date"),
                F.to_date("STOP").alias("discharge_date"),
            ))

    w = Window.partitionBy("cohort_id", "patient").orderBy("admit_date")
    inpt_with_next = (inpt
        .withColumn("next_admit", F.lead("admit_date").over(w))
        .withColumn("days_to_next",
            F.when(F.col("next_admit").isNotNull(),
                   F.datediff(F.col("next_admit"), F.col("discharge_date"))))
        .withColumn("is_readmit_30d",
            F.when((F.col("days_to_next") >= 0) & (F.col("days_to_next") <= 30), 1).otherwise(0)))

    readmissions_30d = (inpt_with_next
        .groupBy(F.col("discharge_date").alias("date"), "cohort_id")
        .agg(F.count("*").alias("index_admissions"),
             F.sum("is_readmit_30d").alias("readmits"))
        .withColumn("readmit_rate",
            F.when(F.col("index_admissions") > 0,
                   F.col("readmits") / F.col("index_admissions")).otherwise(F.lit(0.0))))

    (readmissions_30d.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.exec.readmissions_30d"))
    print("[gold] wrote exec.readmissions_30d")

    # ---------------------------------------------------------------------------
    # exec.cost_summary
    #   Charges & payments rolled up per day from encounters + claims_transactions.
    #   - total_charges    = sum(encounters.TOTAL_CLAIM_COST)  by encounter date
    #   - total_payments   = sum(claims_transactions.PAYMENTS) by transaction FROMDATE
    #   - total_outstanding = sum(claims_transactions.OUTSTANDING) by FROMDATE
    # ---------------------------------------------------------------------------
    charges = (encounters
        .withColumn("date", F.to_date("START"))
        .groupBy("date", "cohort_id")
        .agg(F.sum("TOTAL_CLAIM_COST").alias("total_charges")))

    if "FROMDATE" in claims_tx.columns:
        tx = claims_tx.withColumn("date", F.to_date("FROMDATE"))
    else:
        tx = claims_tx.withColumn("date", F.lit(None).cast("date"))

    payments = (tx
        .groupBy("date", "cohort_id")
        .agg(F.sum(F.coalesce(F.col("PAYMENTS"), F.lit(0.0))).alias("total_payments"),
             F.sum(F.coalesce(F.col("OUTSTANDING"), F.lit(0.0))).alias("total_outstanding")))

    cost_summary = (charges
        .join(payments, ["date", "cohort_id"], "full_outer")
        .na.fill({"total_charges": 0.0, "total_payments": 0.0, "total_outstanding": 0.0})
        .filter(F.col("date").isNotNull()))

    (cost_summary.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.exec.cost_summary"))
    print("[gold] wrote exec.cost_summary")

    # ---------------------------------------------------------------------------
    # exec.quality_measures
    #   A small starter set of clinically meaningful measures. Easy to extend.
    #
    #   diabetes_a1c_control   patients with diabetes whose latest HbA1c <= 7.0
    #   bp_control             patients with HTN whose latest systolic BP <= 140
    #   flu_vaccine_12mo       patients with a flu immunization in last 365d
    #
    #   Numerator/denominator are population counts as of as_of_date, attributed
    #   to as_of_date in the time series.
    # ---------------------------------------------------------------------------
    as_of = F.lit(as_of_date).cast("date")

    # helper: latest observation value per patient for a code
    def _latest_obs(code):
        o = (observations
             .filter(F.col("CODE") == code)
             .filter(F.to_date("DATE") <= as_of)
             .select("cohort_id",
                     F.col("PATIENT").alias("patient"),
                     F.to_date("DATE").alias("obs_date"),
                     F.col("VALUE").cast("double").alias("value")))
        w = Window.partitionBy("cohort_id", "patient").orderBy(F.col("obs_date").desc())
        return o.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")

    # Diabetes A1c control (HbA1c LOINC 4548-4)
    dm_pts = (conditions
              .filter(F.col("CODE").isin("44054006"))   # SNOMED diabetes mellitus type 2
              .select("cohort_id", F.col("PATIENT").alias("patient")).distinct())
    a1c    = _latest_obs("4548-4")
    dm_den = dm_pts.count() if dm_pts.limit(1).count() else 0  # avoids spark error on empty
    dm_join = dm_pts.join(a1c, ["cohort_id", "patient"], "left")
    dm_num  = dm_join.filter(F.col("value") <= 7.0).count()

    # Hypertension BP control (systolic LOINC 8480-6 <= 140)
    htn_pts = (conditions
               .filter(F.col("CODE").isin("59621000", "38341003"))  # essential HTN / HTN disorder
               .select("cohort_id", F.col("PATIENT").alias("patient")).distinct())
    sbp    = _latest_obs("8480-6")
    htn_join = htn_pts.join(sbp, ["cohort_id", "patient"], "left")
    htn_den = htn_pts.count()
    htn_num = htn_join.filter(F.col("value") <= 140).count()

    # Flu vaccine in past 365 days (CVX 140 / 88 / 158 — Synthea uses CVX codes in 'CODE')
    flu_window_start = (dt.date.fromisoformat(as_of_date) - dt.timedelta(days=365)).isoformat()
    imm  = spark.table(f"{SILVER}.immunizations")
    flu  = (imm.filter(F.col("CODE").cast("string").isin("140", "88", "158", "150"))
               .filter((F.to_date("DATE") >= F.lit(flu_window_start)) & (F.to_date("DATE") <= as_of))
               .select("cohort_id", F.col("PATIENT").alias("patient")).distinct())
    adult_pts = (patients
                 .filter(F.datediff(as_of, F.to_date("BIRTHDATE")) / 365.25 >= 18)
                 .select("cohort_id", F.col("Id").alias("patient")))
    flu_den = adult_pts.count()
    flu_num = adult_pts.join(flu, ["cohort_id", "patient"], "inner").count()

    def _qm_row(measure_id, num, den):
        rate = (num / den) if den else 0.0
        return (as_of_date, "ALL", measure_id, num, den, rate)

    quality_rows = [
        _qm_row("diabetes_a1c_control_lt_7", dm_num,  dm_den),
        _qm_row("bp_control_sbp_le_140",     htn_num, htn_den),
        _qm_row("flu_vaccine_12mo_adults",   flu_num, flu_den),
    ]
    quality_measures = spark.createDataFrame(
        quality_rows,
        "date string, cohort_id string, measure_id string, numerator long, denominator long, rate double"
    ).withColumn("date", F.to_date("date"))

    # Per-cohort breakdown for the same measures (best-effort; requires recompute per cohort).
    # For POC simplicity we only emit ALL today; per-cohort can be added by looping over distinct cohort_ids.
    (quality_measures.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.exec.quality_measures"))
    print("[gold] wrote exec.quality_measures")

    # ---------------------------------------------------------------------------
    # exec.equity_demographics
    #   Encounter counts and total cost per (date, cohort_id, race, ethnicity, gender).
    # ---------------------------------------------------------------------------
    enc_pat = (encounters
        .withColumn("date", F.to_date("START"))
        .join(patients.select(F.col("Id").alias("PATIENT"),
                               "cohort_id",
                               "RACE", "ETHNICITY", "GENDER"),
              on=["PATIENT", "cohort_id"], how="left"))

    equity_demographics = (enc_pat
        .groupBy("date", "cohort_id",
                 F.col("RACE").alias("race"),
                 F.col("ETHNICITY").alias("ethnicity"),
                 F.col("GENDER").alias("gender"))
        .agg(F.count("*").alias("encounters"),
             F.sum(F.coalesce(F.col("TOTAL_CLAIM_COST"), F.lit(0.0))).alias("total_cost")))

    (equity_demographics.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.exec.equity_demographics"))
    print("[gold] wrote exec.equity_demographics")

    # ---------------------------------------------------------------------------
    # insights.daily_anomalies
    #   Z-score against a 28-day trailing baseline for two key metrics:
    #   daily encounters and daily total_charges. Severity buckets:
    #     |z| >= 3 -> "high", >= 2 -> "medium", >= 1 -> "low", else "none"
    # ---------------------------------------------------------------------------
    def _anomaly(df, value_col, metric_name):
        w = (Window.partitionBy("cohort_id")
                    .orderBy(F.col("date").cast("timestamp").cast("long"))
                    .rangeBetween(-28 * 86400, -1 * 86400))
        return (df.select("date", "cohort_id", F.col(value_col).cast("double").alias("current"))
                  .withColumn("baseline", F.avg("current").over(w))
                  .withColumn("stddev",   F.stddev_pop("current").over(w))
                  .withColumn("z_score",
                      F.when(F.col("stddev").isNotNull() & (F.col("stddev") > 0),
                             (F.col("current") - F.col("baseline")) / F.col("stddev"))
                       .otherwise(F.lit(0.0)))
                  .withColumn("metric", F.lit(metric_name))
                  .withColumn("severity",
                      F.when(F.abs("z_score") >= 3, "high")
                       .when(F.abs("z_score") >= 2, "medium")
                       .when(F.abs("z_score") >= 1, "low")
                       .otherwise("none")))

    enc_daily = (encounter_volumes
                 .groupBy("date", "cohort_id").agg(F.sum("encounters").alias("encounters")))
    cost_daily = cost_summary.select("date", "cohort_id", "total_charges")

    a1 = _anomaly(enc_daily,  "encounters",    "daily_encounters")
    a2 = _anomaly(cost_daily, "total_charges", "daily_total_charges")

    daily_anomalies = (a1.unionByName(a2)
        .select("date", "cohort_id", "metric", "current", "baseline", "z_score", "severity")
        .filter(F.col("baseline").isNotNull()))

    (daily_anomalies.write
        .format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.insights.daily_anomalies"))
    print("[gold] wrote insights.daily_anomalies")

    import json
    return (json.dumps({"as_of_date": as_of_date, "status": "ok"}))



try:
    _result = _run_body()
except Exception as _e:
    mark(run_id_root, "ALL", stage, "FAILED", error=str(_e))
    raise
mark(run_id_root, "ALL", stage, "SUCCEEDED")
mssparkutils.notebook.exit(_result)
